model 1 :restnet50+lstm

In [ ]:
base_model = keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False
inputs = keras.Input(shape = (224, 224, 3))
x = data_augmentation(inputs, training = True)
x = keras.applications.resnet50.preprocess_input(x)
x = base_model(x, training = False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation = 'relu')(x)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
text_input = keras.Input(shape=(30,), name="text_input")
x_text = layers.Embedding(input_dim = 5000, output_dim=256, mask_zero = True)(text_input)
x_text = layers.LSTM(256, return_sequences=True)(x_text)

combine *restnet50+lstm*

In [ ]:
image_features = layers.RepeatVector(30)(x)

combined = layers.concatenate([image_features, x_text])

x = layers.LSTM(256, return_sequences=True)(combined)

output = layers.TimeDistributed(
    layers.Dense(5000, activation='softmax'))(x)
Resnet50_LSTM_model = keras.Model(
    inputs=[inputs, text_input],
    outputs=output
)

Resnet50_LSTM_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

Resnet50_LSTM_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_input          │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 30, 256)   │  1,280,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 30)        │          0 │ text_input[0][0]  │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 30, 256)   │    525,312 │ embedding[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 30, 256)   │          0 │ dense[0][0]       │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, 30, 1)     │          0 │ not_equal[0][0]   │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zeros_like          │ (None, 30, 256)   │          0 │ lstm[0][0]        │
│ (ZerosLike)         │                   │            │                 

 Total params: 27,990,024 (106.77 MB)

 Trainable params: 4,402,312 (16.79 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

train first model

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
history_Resnet50_LSTM = Resnet50_LSTM_model.fit(
    [x_train, padded],
    y_train,
    validation_data=([x_val, pad_val], y_val),
    epochs=10,
    batch_size=32,
    verbose=1,
    callbacks=[early_stopping]
)

Epoch 1/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 68s 215ms/step - accuracy: 0.7152 - loss: 1.9605 - val_accuracy: 0.7171 - val_loss: 1.5960
Epoch 2/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 196ms/step - accuracy: 0.7534 - loss: 1.4847 - val_accuracy: 0.7677 - val_loss: 1.4173
Epoch 3/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 45s 195ms/step - accuracy: 0.7813 - loss: 1.3252 - val_accuracy: 0.7869 - val_loss: 1.2507
Epoch 4/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 84s 201ms/step - accuracy: 0.7975 - loss: 1.1479 - val_accuracy: 0.8037 - val_loss: 1.0824
Epoch 5/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 197ms/step - accuracy: 0.8133 - loss: 0.9886 - val_accuracy: 0.8177 - val_loss: 0.9599
Epoch 6/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 50s 215ms/step - accuracy: 0.8270 - loss: 0.8753 - val_accuracy: 0.8285 - val_loss: 0.8867
Epoch 7/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 197ms/step - accuracy: 0.8356 - loss: 0.8028 - val_accuracy: 0.8348 - val_loss: 0.8410
Epoch 8/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 199ms/step - accuracy: 0.8431 - loss: 0

save model train

In [ ]:
Resnet50_LSTM_model.save("Resnet50_LSTM_model.keras")
import pickle
with open("tokenizer_en.pkl", "wb") as f:
    pickle.dump(tokenizer_en, f)

Caption Generator Function

In [ ]:
def generate_caption(model, tokenizer_en, image, max_length=30):
    in_text = "<start>"

    for _ in range(max_length):

        sequence = tokenizer_en.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length, padding='post')

        y_pred = model.predict([image, sequence], verbose=0)

        y_pred = np.argmax(y_pred[0, len(in_text.split())-1])

        word = None
        for w, index in tokenizer_en.word_index.items():
            if index == y_pred:
                word = w
                break

        if word is None:
            break

        in_text += " " + word

        # 👇 هنا مكان الكود بتاعك
        if word == "<end>" or len(in_text.split()) > 20:
            break

    return in_text

test first model

In [ ]:
test_loss, test_accuracy = Resnet50_LSTM_model.evaluate([x_test, pad_test], y_test, verbose = 1)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 149ms/step - accuracy: 0.8464 - loss: 0.7680
Test Loss: 0.7679556012153625
Test Accuracy: 0.8463904857635498


train in arabic

In [ ]:
base_model = keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False
inputs = keras.Input(shape = (224, 224, 3))
x = data_augmentation(inputs, training = True)
x = keras.applications.resnet50.preprocess_input(x)
x = base_model(x, training = False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation = 'relu')(x)

In [ ]:
text_input = keras.Input(shape=(30,), name="text_input")
x_text = layers.Embedding(input_dim = 5000, output_dim=256, mask_zero = True)(text_input)
x_text = layers.LSTM(256, return_sequences=True)(x_text)

In [ ]:
image_features = layers.RepeatVector(30)(x)

combined = layers.concatenate([image_features, x_text])

x = layers.LSTM(256, return_sequences=True)(combined)

output = layers.TimeDistributed(
    layers.Dense(5000, activation='softmax'))(x)
Resnet50_LSTM_model_ar = keras.Model(
    inputs=[inputs, text_input],
    outputs=output
)

Resnet50_LSTM_model_ar.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

Resnet50_LSTM_model_ar.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ sequential[1][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ sequential[1][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ sequential[1][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_input          │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 30, 256)   │  1,280,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 30)        │          0 │ text_input[0][0]  │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    524,544 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 30, 256)   │    525,312 │ embedding_1[0][0… │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_2     │ (None, 30, 256)   │          0 │ dense_2[0][0]     │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_1       │ (None, 30, 1)     │          0 │ not_equal_1[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zeros_like_1        │ (None, 30, 256)   │          0 │ lstm_2[0][0]    

 Total params: 27,990,024 (106.77 MB)

 Trainable params: 4,402,312 (16.79 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
history_Resnet50_LSTM_ar = Resnet50_LSTM_model_ar.fit(
    [x_train_ar, ar_train_pad],
    y_ar_train,
    validation_data = (
        [x_val_ar, ar_val_pad],
        y_ar_val),
    epochs = 10,
    batch_size = 32,
    callbacks=[early_stopping]
)

Epoch 1/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 60s 213ms/step - accuracy: 0.7496 - loss: 1.9877 - val_accuracy: 0.7529 - val_loss: 1.6075
Epoch 2/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 45s 194ms/step - accuracy: 0.7762 - loss: 1.5015 - val_accuracy: 0.7932 - val_loss: 1.4451
Epoch 3/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 196ms/step - accuracy: 0.7966 - loss: 1.3483 - val_accuracy: 0.8022 - val_loss: 1.3120
Epoch 4/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 196ms/step - accuracy: 0.8085 - loss: 1.1983 - val_accuracy: 0.8102 - val_loss: 1.1862
Epoch 5/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 196ms/step - accuracy: 0.8197 - loss: 1.0684 - val_accuracy: 0.8204 - val_loss: 1.0935
Epoch 6/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 45s 195ms/step - accuracy: 0.8283 - loss: 0.9710 - val_accuracy: 0.8274 - val_loss: 1.0290
Epoch 7/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 196ms/step - accuracy: 0.8355 - loss: 0.8940 - val_accuracy: 0.8342 - val_loss: 0.9777
Epoch 8/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 46s 195ms/step - accuracy: 0.8425 - loss: 0

save model

In [ ]:
Resnet50_LSTM_model_ar.save("Resnet50_LSTM_model_ar.keras")
import pickle
with open("tokenizer_ar.pkl", "wb") as f:
    pickle.dump(tokenizer_ar, f)

function

In [ ]:
def generate_caption(model, tokenizer_ar, image, max_length=30):
    in_text = "<start>"

    for _ in range(max_length):

        sequence = tokenizer_ar.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length, padding='post')

        y_pred = model.predict([image, sequence], verbose=0)

        y_pred = np.argmax(y_pred[0, len(in_text.split())-1])

        word = None
        for w, index in tokenizer_ar.word_index.items():
            if index == y_pred:
                word = w
                break

        if word is None:
            break

        in_text += " " + word

        # 👇 هنا مكان الكود بتاعك
        if word == "<end>" or len(in_text.split()) > 20:
            break

    return in_text

test

In [ ]:
test_loss, test_accuracy = Resnet50_LSTM_model_ar.evaluate([x_test_ar, ar_test_pad], y_ar_test, verbose = 1)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

50/50 ━━━━━━━━━━━━━━━━━━━━ 7s 147ms/step - accuracy: 0.8457 - loss: 0.8792
Test Loss: 0.8792107105255127
Test Accuracy: 0.8457001447677612
